# <a id='toc1_'></a>[Capturing Alternate Abbreviation Symbols Experiment](#toc0_)

Goal of this experiment is to see how well Claude can match my annotations of gene alias pairs in literary context

**Table of contents**<a id='toc0_'></a>    
- [Capturing Alternate Abbreviation Symbols Experiment](#toc1_)    
    - [Import manually curated table](#toc1_1_1_)    
    - [Add the official gene name for each sample from HGNC](#toc1_1_2_)    
    - [Add the abstract and full text for each sample](#toc1_1_3_)    
      - [Not all contexts are programmatically accessible](#toc1_1_3_1_)    
    - [Using details from each sample, generate a custom prompt for each sample](#toc1_1_4_)    
      - [Prompt with just the abstract as context](#toc1_1_4_1_)    
      - [Prompt with full text as context](#toc1_1_4_2_)    
      - [Prompt with full text as context but abstract if full is not available](#toc1_1_4_3_)    
      - [Prompt that doesn't require context](#toc1_1_4_4_)    
    - [Run prompt through Claude](#toc1_1_5_)    
      - [Run a test with 2 samples (available context, prioritizing full text)](#toc1_1_5_1_)    
      - [Run a test with 20 samples (available context, prioritizing full text)](#toc1_1_5_2_)    
    - [Run all samples (available context, prioritizing full text)](#toc1_1_6_)    
    - [RUN SAMPLE OF 30 WO CONTEXT](#toc1_1_7_)    
    - [Run all samples with prompt that doesnt use context](#toc1_1_8_)    
    - [LLM-as-a-judge](#toc1_1_9_)    
    - [Analysis](#toc1_1_10_)    
      - [Functions](#toc1_1_10_1_)    
      - [Analysis (20 samples, available context, prioritizing full text)](#toc1_1_10_2_)    
      - [Analyze (available context, prioritizing full text)](#toc1_1_10_3_)    
      - [Analyze the results from using prompt with no context](#toc1_1_10_4_)    
    - [Engineering with False samples](#toc1_1_11_)    
    - [Evaluator prompt work](#toc1_1_12_)    
      - [Analyze results from running evaluator prompt](#toc1_1_12_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [ ]:
import ssl
import certifi
import os

os.environ["SSL_CERT_FILE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

from collections.abc import Mapping
from enum import StrEnum
from pathlib import Path
from typing import Any

import polars as pl
from pydantic import BaseModel, ConfigDict
from tqdm.notebook import tqdm
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import BasePromptTemplate, build_empty_registry
from wags_llm.services import StructuredTaskRunner

: 

In [ ]:
import pandas as pd
import requests
from tqdm import tqdm
tqdm.pandas()
import boto3
import json
from functools import lru_cache
import numpy as np
import re
from rapidfuzz.distance import LCSseq

: 

### Functions

In [ ]:
def get_gene_name(hgnc_id):
    """ Retreive official gene name from HGNC for each sample in the set.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol of each sample
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    url = f"https://rest.genenames.org/fetch/hgnc_id/{hgnc_id}"
    headers = {"Accept": "application/json"}  # Request JSON format
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        try:
            gene_name = data['response']['docs'][0]['name']
            return gene_name
        except IndexError:
            return f"No gene found for HGNC ID {hgnc_id}"
    else:
        return f"Error: {response.status_code}"

In [ ]:
gene_cache = {}

def get_gene_name_cached(hgnc_id):
    """ Retrieve the official gene name from HGNC using a cache to avoid repeated API requests for the same HGNC ID.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    if hgnc_id not in gene_cache:
        gene_cache[hgnc_id] = get_gene_name(hgnc_id)
    return gene_cache[hgnc_id]

In [ ]:
def generate_alt_abbrev_prompt_w_no_context(alias_symbol, hgnc_id, primary_gene_symbol, name):
    """ Generate an LLM prompt for curating alias gene symbols and determining if their relationship to an official HGNC gene record is an alternate abbreviation.

    The generated prompt instructs the model to:
    - Classify the relationship as an Alternate Abbreviation or not

    :param alias_symbol: The alias gene symbol to be evaluated
    :param hgnc_id: The HGNC record ID associated with the primary gene
    :param primary_gene_symbol: The official HGNC gene symbol
    :param name: The official HGNC gene name
    :return: A formatted prompt string for use with a language model
    """
    prompt = f"""Role:You are a biomedical gene nomenclature curator trained in HGNC gene identity and alias symbol resolution.
    You will get fired if your accuracy is less than 95%.
    
    Background: An alternate abbreviation is when an alias symbol represents the same official gene name or the 
    primary gene symbol but with different letters. If the alias symbol is representing a different description or a previous name of
    the gene then it is not an alternate abbreviation. Be careful with alias symbols that seem to be shortened versions
    of the primary gene symbol, they may be a family name and therefore not an alternate abbreviation. Alias symbols
    have extra characters may be alternate abbreviations unless they have characters that are not present in the official 
    gene name provided in the prompt. Keep in mind, a gene name with a number after the name is not the same gene as a gene name 
    with no numbers or different numbers after the name.

    Task: Determine whether the alias symbol is an abbreviation variant of the official HGNC gene name. Give a one sentence reason for
    your determination.

    Here is the gene information:
    Alias gene symbol: {alias_symbol}
    HGNC record ID: {hgnc_id}
    Primary gene symbol: {primary_gene_symbol}
    Official HGNC gene name: {name}
        
    Rules:
    If the alias symbol is not an alternate abbreviation, return:
    "alternate_abbreviation": "FALSE"
    If the alias symbol is an alternate abbreviation, return:
    "alternate_abbreviation": "TRUE"

    Output (Strict):
    Output only one JSON object. No markdown.
    {{
    "reason": statement,
    "alternate_abbreviation": "TRUE or FALSE"
    }}

    """

    return prompt

In [ ]:
# Initialize the Bedrock Runtime client
session = boto3.Session(profile_name="dev-account")
bedrock = session.client("bedrock-runtime", region_name="us-east-2")

# Replace with your actual inference profile ID or ARN
INFERENCE_PROFILE_ID = "us.anthropic.claude-3-5-sonnet-20240620-v1:0"


def query_claude_sonnet(prompt: str) -> str:
    """ Send a prompt to the Claude Sonnet model via Amazon Bedrock and return the generated response text.

    :param prompt: The input prompt to send to the Claude Sonnet model
    :return: The model-generated response text, or an error message string
    """

    body = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1024,
        "temperature": 0.0,
        "anthropic_version": "bedrock-2023-05-31"
    }

    try:
        response = bedrock.invoke_model(
            body=json.dumps(body),
            modelId=INFERENCE_PROFILE_ID,
            contentType="application/json",
            accept="application/json"
        )
        response_body = json.loads(response["body"].read())
        return response_body["content"][0]["text"]
    except Exception as e:
        return f"[Error] {str(e)}"

In [ ]:
def lcs_similarity(s1, s2):
    return LCSseq.normalized_similarity(s1, s2)

def has_extra_characters(alias, gene_name):
    norm_alias = re.sub(r'[^A-Z0-9]', '', alias.upper())

    alias_set = set(norm_alias)
    name_set = set(gene_name)

    extra_chars = alias_set - name_set

    return len(extra_chars)

def should_call_llm(alias_symbol, primary_gene_symbol, gene_name, threshold=0.20):
    """Determine whether an alias symbol should be sent to the LLM for further evaluation.

    Applies a series of heuristic filters to the alias symbol
    - checks for excluded prefixes ("HSA-", this is not an alternate abbreviation alias 
    because it is an identifier being used as an alias)
    - extra characters in the alias, not in the gene name
    - longest common subsequence (LCS) similarity to the primary gene symbol to be at least 20%

    :param alias_symbol: The alias gene symbol being evaluated
    :param primary_gene_symbol: The official primary gene symbol
    :param gene_name: The full gene name associated with the gene
    :param threshold: Minimum LCS similarity score required to pass filtering
    :return: A tuple containing:
        - bool: Whether the alias should be sent to the LLM
        - float: The computed LCS similarity score
        - str: The reason code for the decision
    """

    alias = alias_symbol.upper()
    primary = primary_gene_symbol.upper()
    name = gene_name.upper()

    if alias.startswith("HSA-"):
        return False, 0, "hsa_prefix"

    extra_count = has_extra_characters(alias_symbol, name)

    # block only if ≥2 extra characters
    if extra_count >= 2:
        return False, 0, "extra_characters"

    score = lcs_similarity(alias, primary)

    if score < threshold:
        return False, score, "low_lcs_similarity"

    return True, score, "passed_filters"

In [ ]:
def extract_json_block(s):
    if pd.isna(s) or not str(s).strip():
        return np.nan

    s = str(s)

    stack = []
    start = None

    for i, ch in enumerate(s):
        if ch in '{[':
            stack.append(ch)
            if start is None:
                start = i
        elif ch in '}]' and stack:
            stack.pop()
            if not stack:
                return s[start:i+1]

    return np.nan


def parse_response(x):
    if pd.isna(x) or not str(x).strip():
        return np.nan

    s = str(x)

    block = extract_json_block(s)
    if not block:
        return np.nan

    try:
        return json.loads(block)
    except json.JSONDecodeError:
        return np.nan

In [ ]:
def extract_alternate_abbreviation(d):
    if not isinstance(d, dict):
        return np.nan

    val = d.get("alternate_abbreviation", None)

    if val is None or val == "":
        return np.nan

    return str(val).strip().upper() == "TRUE"

In [ ]:
def extract_reason(d):
    if not isinstance(d, dict):
        return np.nan

    val = d.get("reason", None)

    if val is None or val == "":
        return np.nan

    return str(val).strip()

In [ ]:
def create_analysis_summary(column: pd.Series):
    """ Summarize value counts with:
      - True/False percentages over non-NaN denominator
      - NaN percentage over total rows

    :param column: Dataframe and column to analyze (Example: df["column"])
    :return: dataframe with summarized results
    """
    counts = (
        column.value_counts(dropna=False)
              .rename("numerator")
              .rename_axis("consensus_w_curator")
    )

    non_nan_denominator = column.notna().sum()
    total_denominator = len(column)

    return (
        counts
        .reset_index()
        .assign(
            denominator=lambda df: df["consensus_w_curator"].apply(
                lambda x: total_denominator if pd.isna(x) else non_nan_denominator
            ),
            percentage=lambda df: df["numerator"] / df["denominator"] * 100
        )
    )

In [ ]:
def consolidate_bool(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    if s.any():
        return True
    return False

In [ ]:
def boolean_confusion_matrix(df, gt_col, pred_col):
    """
    Compute a boolean confusion matrix in the traditional matrix style.

    Parameters:
    - df: pandas DataFrame
    - gt_col: str, name of the ground truth column
    - pred_col: str, name of the predicted column

    Returns:
    - confusion_matrix: pandas DataFrame with row/column labels
    """
    gt = df[gt_col]
    pred = df[pred_col]

    # Keep only valid predictions
    mask = pred.notna()
    gt = gt[mask]
    pred = pred[mask]

    # Build crosstab
    cm = pd.crosstab(gt, pred, dropna=False)

    # Reindex to ensure consistent order
    cm = cm.reindex(index=[False, True], columns=[False, True], fill_value=0)

    # Set row and column labels for display
    cm.index.name = "Ground Truth"
    cm.columns.name = "Predicted"

    return cm

In [ ]:
def analyze_alt_abbrev_results(    
    df,
    extract_evidence=True,
    consolidate=True,
    groupby_col="pubtator3_pmid",
):
    analysis_df = df.copy()

    # Extract and parse
    analysis_df["alt_abbrev_response_extracted"] = (
        analysis_df["alt_abbrev_response"].apply(extract_json_block)
    )

    analysis_df["alt_abbrev_response_parsed"] = (
        analysis_df["alt_abbrev_response_extracted"].apply(parse_response)
    )
    analysis_df["response_alternate_abbreviation"] = (
        analysis_df["alt_abbrev_response_parsed"]
        .apply(extract_alternate_abbreviation)
    )
    analysis_df["response_reason"] = (
    analysis_df["alt_abbrev_response_parsed"]
    .apply(extract_reason)
    )

    ## optional: extract evidence
    if extract_evidence:
        analysis_df["response_evidence"] = (
            analysis_df["alt_abbrev_response_parsed"]
            .apply(lambda d: d.get("evidence") if isinstance(d, dict) else None)
        )

    # Optional: Consolidate per PMID
    if consolidate:
        consolidated_df = (
            analysis_df
            .groupby(groupby_col, as_index=False)
            .agg({
                "alternate_abbreviation_status_w_no_context": "first",
                "response_alternate_abbreviation": consolidate_bool,
            })
        )
    else:
        consolidated_df = analysis_df


    # Compare to manual curation
    consolidated_df["relationship_type_matches"] = (
        consolidated_df["alternate_abbreviation_status_w_no_context"]
        == consolidated_df["response_alternate_abbreviation"]
    )

    # Summarize
    relationship_type_match_analysis_summary = create_analysis_summary(
        consolidated_df["relationship_type_matches"]
    )

    # Create confusion matrix
    cm = boolean_confusion_matrix(
        consolidated_df,
        "alternate_abbreviation_status_w_no_context",
        "response_alternate_abbreviation",
    )

    TN = cm.loc[False, False]
    FP = cm.loc[False, True]
    FN = cm.loc[True, False]
    TP = cm.loc[True, True]

    # Compute metrics
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    metrics_df = pd.DataFrame({
        "precision": [precision],
        "recall": [recall],
        "f1": [f1],
        "TP": [TP],
        "FP": [FP],
        "TN": [TN],
        "FN": [FN],
    })


    return analysis_df, consolidated_df, relationship_type_match_analysis_summary, cm, metrics_df


### <a id='toc1_1_1_'></a>[Import manually curated table](#toc0_)

This is a file where I manually annotated gene alias pairs by answering the following questions:
- Does this alias symbol represent this gene in this publication?
- How does this alias symbol represent this gene in this publication

Reference alternate_abbreviation_annotation_doc for content description

In [ ]:
alt_abbrev_annotation_wo_context_df = pd.read_csv(
    "../input/Alternate Abbreviation Symbol Capture Without Context- Manually Curated Set.csv", encoding="latin1", skip_blank_lines=True, true_values=["TRUE", "T"],
    false_values=["FALSE", "F"])

### <a id='toc1_1_2_'></a>[Add the official gene name for each sample from HGNC](#toc0_)

In [ ]:
alt_abbrev_annotation_wo_context_df["gene_name"] = alt_abbrev_annotation_wo_context_df["HGNC_ID"].apply(get_gene_name_cached)
alt_abbrev_annotation_wo_context_df

In [ ]:
alt_abbrev_annotation_wo_context_df.to_excel('../output/alt_abbrev_annotation_wo_context_df.xlsx',index=False)

In [ ]:
alt_abbrev_annotation_wo_context_df.to_csv('../output/alt_abbrev_annotation_wo_context_df.csv',index=False)

In [ ]:
first_ten_alt_abbrev_annotation_wo_context_df = alt_abbrev_annotation_wo_context_df.head(10)

In [ ]:
first_ten_alt_abbrev_annotation_wo_context_df.to_csv('../output/first_ten_alt_abbrev_annotation_wo_context_df.csv',index=False)

In [ ]:
SAMPLE_PATH = Path(
    "../output/first_ten_alt_abbrev_annotation_wo_context_df.csv"
)

In [ ]:
PROMPT_NAME = "alias_symbol_annotation:alternate_abbreviation"
PROMPT_VERSION = "v1"


class AlternateAbbreviationPromptV1(BasePromptTemplate):
    """Version 1 prompt for predicting alternate abbreviation relationships."""

    version = PROMPT_VERSION
    name = PROMPT_NAME

    def build_system_prompt(self) -> str:
        """Build the system prompt for predicting whether an alias is an alternate abbreviation of the primary gene symbol.

        :returns: System prompt text.
        """
        return (
            "Role: You are a biomedical gene nomenclature curator trained in HGNC gene identity and alias symbol resolution.\n"
            "You will get fired if your accuracy is less than 95%.\n"
            "Background:\n"
            "An alternate abbreviation is when an alias symbol represents the same official gene name or the\n"
            "primary gene symbol but with different letters. If the alias symbol is representing a different description or a previous name of\n"
            "the gene then it is not an alternate abbreviation. Be careful with alias symbols that seem to be shortened versions\n"
            "of the primary gene symbol, they may be a family name and therefore not an alternate abbreviation. Alias symbols\n"
            "have extra characters may be alternate abbreviations unless they have characters that are not present in the official\n"
            "gene name provided in the prompt. Keep in mind, a gene name with a number after the name is not the same gene as a gene name\n"
            "with no numbers or different numbers after the name.\n"
            "Task:\n"
            "Determine whether the alias symbol is an abbreviation variant of the official HGNC gene name.\n"
        )

    def build_user_prompt(
        self,
        payload: Mapping[str, Any],
    ) -> str:
        """Build the user prompt for a single alias symbol.

        :param payload: The alias symbol, HGNC ID, primary gene symbol, official gene name to be evaluated, 
        :returns: User prompt text
        """
        return (
            f"Alias Symbol: {payload['gene_symbol']}\n"
            f"Primary Gene Symbol: {payload['primary_gene_symbol']}\n"
            f"Official Gene Name: {payload['gene_name']}\n"
            f"HGNC ID: {payload['hgnc_id']}\n"
        )

In [ ]:
class AlternateAbbreviationPrediction(StrEnum):
    """Predict whether an alias is an alternate abbreviation of the primary gene symbol."""

    TRUE = "is alternate abbreviation"
    FALSE = "is not alternate abbreviation"
    # UNCLEAR = "unclear" maybe introduce this category later to help narrow down cases for human review


class AlternateAbbreviationPredictionResult(BaseModel):
    """Model for LLM and human curator result for determining whether an
    alias symbol corresponds to the primary gene symbol.
    """

    model_config = ConfigDict(extra="forbid", use_enum_values=True)

    alt_abbrev_prediction: AlternateAbbreviationPrediction

In [ ]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 150
TEMPERATURE = 0.0


def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM alternate abbreviation alias annotator

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(AlternateAbbreviationPromptV1())
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )


task_runner = build_llm_task_runner(
    MODEL_ID, REGION_NAME, PROFILE_NAME, MAX_TOKENS, TEMPERATURE
)

In [ ]:
response_model = AlternateAbbreviationPredictionResult
df = pl.read_csv(SAMPLE_PATH, separator=",")

annotation_values = []
error_messages = []

In [ ]:
for _i, row in enumerate(
    tqdm(df.iter_rows(named=True), total=df.height, desc="Processing rows")
):
    gene_symbol = row["gene_symbol"]
    primary_gene_symbol = row["primary_gene_symbol"]
    gene_name = row["gene_name"]
    hgnc_id = row["HGNC_ID"]

    try:
        task_result = task_runner.execute(
            prompt_name=PROMPT_NAME,
            prompt_version=PROMPT_VERSION,
            payload={
                "gene_symbol": gene_symbol,
                "primary_gene_symbol": primary_gene_symbol,
                "gene_name": gene_name,
                "hgnc_id": hgnc_id,
            },
            response_model=response_model,
        )

    except Exception as e:
        annotation_value = "Error"
        error_msg = str(e)

    else:
        annotation = response_model.model_validate(task_result)
        annotation_value = annotation.alt_abbrev_prediction
        error_msg = None

    annotation_values.append(annotation_value)
    error_messages.append(error_msg)

In [ ]:
error_messages

In [ ]:
annotation_values

In [ ]:
df = df.with_columns(
    [
        pl.Series("LLM Annotation", annotation_values),
        pl.Series("LLM error message", error_messages),
    ]
)

In [ ]:
df

### <a id='toc1_1_4_'></a>[Using details from each sample, generate a custom prompt for each sample](#toc0_)

In [ ]:
alt_abbrev_annotation_with_prompt_no_context_df = alt_abbrev_annotation_wo_context_df.copy()
alt_abbrev_annotation_with_prompt_no_context_df['alt_abbrev_prompt'] = None
for idx, row in alt_abbrev_annotation_with_prompt_no_context_df.iterrows():
    alias_symbol = row['gene_symbol']
    primary_gene_symbol = row['primary_gene_symbol']
    name = row['gene_name']
    hgnc_id = row['HGNC_ID']
    prompt = generate_alt_abbrev_prompt_w_no_context(alias_symbol, hgnc_id, primary_gene_symbol, name)

    alt_abbrev_annotation_with_prompt_no_context_df.at[idx,'alt_abbrev_prompt'] = prompt    



alt_abbrev_annotation_with_prompt_no_context_df.head()

### <a id='toc1_1_5_'></a>[Run prompt through Claude](#toc0_)

login via aws sso login in terminal

### <a id='toc1_1_7_'></a>[RUN SAMPLE OF 30 WO CONTEXT](#toc0_)

In [ ]:
test30_alt_abbrev_annotation_with_prompt_no_context_df = (
    alt_abbrev_annotation_with_prompt_no_context_df
        .sample(n=30, random_state=44)
)

In [ ]:
test30_alt_abbrev_annotation_with_prompt_no_context_df

In [ ]:
# #Comment out after run!
# test30_alt_abbrev_annotation_with_prompt_no_context_df['alt_abbrev_response'] = None

# for idx, row in test30_alt_abbrev_annotation_with_prompt_no_context_df.iterrows():

#     test30_alt_abbrev_annotation_with_prompt_no_context_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

In [ ]:
# #Comment out after run!
# test30_alt_abbrev_annotation_with_prompt_no_context_df.to_excel('../output/test30_alt_abbrev_annotation_with_prompt_no_context_df.xlsx',index=False)

In [ ]:
test30_alt_abbrev_annotation_with_prompt_no_context_df = pd.read_excel("../output/test30_alt_abbrev_annotation_with_prompt_no_context_df.xlsx")

### <a id='toc1_1_8_'></a>[Run all samples with prompt that doesnt use context](#toc0_)

In [ ]:
alt_abbrev_annotation_with_prompt_no_context_df[
    ["call_llm", "lcs_score", "filter_reason"]
] = (
    alt_abbrev_annotation_with_prompt_no_context_df.apply(
        lambda row: should_call_llm(
            row["gene_symbol"],
            row["primary_gene_symbol"],
            row["gene_name"]
        ),
        axis=1,
        result_type="expand"
    )
)

In [ ]:
# #Comment out after run!
# alt_abbrev_annotation_with_prompt_no_context_df['alt_abbrev_response'] = None

# for idx, row in alt_abbrev_annotation_with_prompt_no_context_df.iterrows():

#     if row['call_llm']:
#         response = query_claude_sonnet(row['alt_abbrev_prompt'])
#     else:
#         response = '{"alternate_abbreviation": "FALSE"}'

#     alt_abbrev_annotation_with_prompt_no_context_df.at[
#         idx, 'alt_abbrev_response'
#     ] = response

#     print(f'{idx} Done')

In [ ]:
##Comment out after run!
# alt_abbrev_annotation_with_prompt_no_context_df.to_excel('../output/alt_abbrev_annotation_with_prompt_no_context_df.xlsx',index=False)

In [ ]:
alt_abbrev_annotation_with_prompt_no_context_df = pd.read_excel("../output/alt_abbrev_annotation_with_prompt_no_context_df.xlsx")

### <a id='toc1_1_10_'></a>[Analysis](#toc0_)

#### <a id='toc1_1_10_4_'></a>[Analyze the results from using prompt with no context](#toc0_)

In [ ]:
analysis_alt_abbrev_annotation_with_prompt_no_context_df, _, relationship_type_match_analysis_summary, cm, metrics_df = analyze_alt_abbrev_results(alt_abbrev_annotation_with_prompt_no_context_df, extract_evidence=False, consolidate=False
)

In [ ]:
relationship_type_match_analysis_summary

In [ ]:
cm

In [ ]:
metrics_df

### <a id='toc1_1_11_'></a>[Engineering with False samples](#toc0_)

In [ ]:
false_positives = analysis_alt_abbrev_annotation_with_prompt_no_context_df.loc[(analysis_alt_abbrev_annotation_with_prompt_no_context_df["alternate_abbreviation_status_w_no_context"] == False) & (analysis_alt_abbrev_annotation_with_prompt_no_context_df["response_alternate_abbreviation"] == True)]
false_positives

In [ ]:
false_positives_list = false_positives["gene_symbol"].tolist()

In [ ]:
df_small_positives = alt_abbrev_annotation_with_prompt_no_context_df[alt_abbrev_annotation_with_prompt_no_context_df["gene_symbol"].isin(false_positives_list)]

In [ ]:
false_negatives = analysis_alt_abbrev_annotation_with_prompt_no_context_df.loc[(analysis_alt_abbrev_annotation_with_prompt_no_context_df["alternate_abbreviation_status_w_no_context"] == True) & (analysis_alt_abbrev_annotation_with_prompt_no_context_df["response_alternate_abbreviation"] == False)]
false_negatives

In [ ]:
false_negatives_list = false_negatives["gene_symbol"].tolist()

In [ ]:
df_small_negatives = alt_abbrev_annotation_with_prompt_no_context_df[alt_abbrev_annotation_with_prompt_no_context_df["gene_symbol"].isin(false_negatives_list)]

In [ ]:
# #Comment out after run!
# df_small_positives['alt_abbrev_response'] = None

# for idx, row in df_small_positives.iterrows():

#     df_small_positives.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

In [ ]:
# #Comment out after run!
# df_small_positives.to_excel('../output/df_small_positives.xlsx',index=False)

In [ ]:
df_small_positives = pd.read_excel("../output/df_small_positives.xlsx")

In [ ]:
# #Comment out after run!
# df_small_negatives['alt_abbrev_response'] = None

# for idx, row in df_small_negatives.iterrows():

#     df_small_negatives.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

In [ ]:
# #Comment out after run!
# df_small_negatives.to_excel('../output/df_small_negatives.xlsx',index=False)

In [ ]:
df_small_negatives = pd.read_excel("../output/df_small_negatives.xlsx")

In [ ]:
df_small = pd.concat([df_small_negatives, df_small_positives], ignore_index=True)

In [ ]:
analysis_df_small, _, small_relationship_type_match_analysis_summary, small_cm, small_metrics_df = analyze_alt_abbrev_results(df_small, extract_evidence=False, consolidate=False
)

In [ ]:
small_relationship_type_match_analysis_summary

In [ ]:
small_cm

In [ ]:
small_metrics_df

In [ ]:
analysis_alt_abbrev_annotation_with_prompt_no_context_df.loc[analysis_alt_abbrev_annotation_with_prompt_no_context_df["gene_symbol"] == "GAB"]